In [ ]:
# =============================================================================
# STAGE 2 — STEP 0: test-side text preparation + real-speed check
# Amazon ML Challenge 2026 — Business Entity Resolution
# =============================================================================
# Run in the notebook that holds /kaggle/working/ckpt_test (the test blocking
# output). Needs no labels, so it can run while the train blocking commits.
#
# PART 1  Cleans every test name/address once (identical code to Notebook 2)
#         and saves compact text columns aligned to each country's qids/pids:
#             /kaggle/working/prep_test/<country>/q.parquet, p.parquet
#         Re-running skips files that already exist.
# PART 2  Times the real Stage 2 operations on real test pairs, on this machine.
# PART 3  Projects the Step 1 (train) and Step 2 (test) runtimes from PART 2.
#
# SETTINGS: keep this notebook's current settings. No GPU is used.
#           Internet ON is only needed if rapidfuzz must be installed/upgraded.
# RUNTIME:  ~15 min. Disk: ~1-1.5 GB for prep_test/.
# =============================================================================

DATA_ROOT = "/kaggle/input/datasets/mahimasunilbachhav/amazon-ml-challenge/student_resource/dataset"
WORK = "/kaggle/working"

BENCH_PAIRS = 1_000_000          # real pairs used for the string-feature speed test
BENCH_GROUP_ROWS = 10_000_000    # real pairs used for the group-feature speed test
N_KEEP_OPTIONS = (20, 30)        # candidates kept per entity by the pre-filter
MATCHER_ROUNDS = 600             # expected boosting rounds (early stopping decides)
TRAIN_S1 = 2_206_821             # train_source1 rows (from EDA)
TRAIN_POOL = 10_320_219          # train S2+S3 rows (from EDA)

import gc
import json
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import lightgbm as lgb

try:
    from rapidfuzz.process import cpdist
except ImportError:
    print("installing/upgrading rapidfuzz (needs Internet ON) ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "rapidfuzz"], check=True)
    try:
        from rapidfuzz.process import cpdist
    except ImportError:
        raise SystemExit("rapidfuzz was upgraded: restart the session, then rerun this cell.")
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler

DATA_ROOT, WORK = Path(DATA_ROOT), Path(WORK)
CKPT = WORK / "ckpt_test"
PREP = WORK / "prep_test"
PREP.mkdir(exist_ok=True)
assert CKPT.exists(), f"{CKPT} not found: the test blocking output must be in /kaggle/working"
T_START = time.time()


def log(msg):
    print(f"  [{(time.time()-T_START)/60:5.1f} min] {msg}", flush=True)


def mem(tag=""):
    rss = psutil.Process().memory_info().rss / 1e9
    print(f"    [MEM] {tag:<26} RAM={rss:5.1f} GB  free={psutil.virtual_memory().available/1e9:5.1f} GB",
          flush=True)


COLS = ["entity_id", "business_name", "business_address", "country"]


def load_src(path):
    """Read a source TSV. sep='\t' is mandatory: fields contain commas."""
    return pd.read_csv(path, sep="\t", dtype=str, names=COLS, header=0, keep_default_na=False)


# ---- cleaning: copied verbatim from Notebook 2 so both stages agree ----------
LEGAL_RE = (r"\b(llc|l\.l\.c|inc|corp|corporation|ltd|limited|pvt|private|"
            r"llp|lp|pllc|sas|sarl|eurl|sasu|snc|company)\b")

ADDR_ABBR = [
    (r"\brd\b", "road"), (r"\bst\b", "street"), (r"\bdr\b", "drive"),
    (r"\bave\b", "avenue"), (r"\bblvd\b", "boulevard"), (r"\bln\b", "lane"),
    (r"\bct\b", "court"), (r"\bcir\b", "circle"), (r"\bhno\b", "house number"),
    (r"\bno\b", "number"), (r"\bopp\b", "opposite"), (r"\bnr\b", "near"),
    (r"\bdist\b", "district"), (r"\bflr\b", "floor"),
]


def clean_text(s):
    """Shared cleaning. Non-Latin scripts are dropped here; L3 reads raw text."""
    s = s.fillna("").astype(str).str.lower()
    s = (s.str.normalize("NFKD")
          .str.encode("ascii", errors="ignore")
          .str.decode("ascii"))
    s = s.str.replace(r"##\s*", "", regex=True)
    s = s.str.replace(r"\bwww\.", "", regex=True)
    s = s.str.replace(r"\.(com|net|org|co|in|fr|uk|biz|info|io)\b", "", regex=True)
    s = s.str.replace(r"^[\s\-\.\,\!\@\#\$\%\^\&\*\>\<\'\"/\\]+", "", regex=True)
    s = s.str.replace("&", " and ", regex=False)
    s = s.str.replace(r"\bn/?a\b", " ", regex=True)
    s = s.str.replace(r"[^\w\s]", " ", regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()


def clean_name(s):
    """Name: shared cleaning + legal suffixes + repeated words (FH FH -> FH)."""
    s = clean_text(s)
    s = s.str.replace(LEGAL_RE, " ", regex=True)
    s = s.str.replace(r"\b(\w+)\s+\1\b", r"\1", regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()


def clean_addr(s):
    """Address: shared cleaning + abbreviation expansion."""
    s = clean_text(s)
    for pat, rep in ADDR_ABBR:
        s = s.str.replace(pat, rep, regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()


# ---- extra text columns for Stage 2 -------------------------------------------
# Street-type words and French articles are skipped when forming the address key,
# so "15 Rue de la Hoxie" -> "15_hoxie" (not "15_rue"), matching "351 Hoxie Avenue"
# style keys. This matters most for France, which training never sees.
SKIP_AFTER_NUMBER = {
    "road", "street", "drive", "avenue", "boulevard", "lane", "court", "circle",
    "way", "highway", "rue", "impasse", "allee", "chemin", "place", "route", "quai",
    "cours", "square", "de", "du", "des", "la", "le", "les", "l", "d",
}


def addr_key_string(caddr):
    """House number + the first following token that isn't a street word/article.
    Returned as a space-joined, sorted string so rapidfuzz can compare two of them."""
    toks = caddr.split()
    keys = set()
    for i, a in enumerate(toks):
        if a.isdigit():
            a2 = a.lstrip("0")
            j = i + 1
            while j < len(toks) and toks[j] in SKIP_AFTER_NUMBER:
                j += 1
            if a2 and j < len(toks):
                b = toks[j].lstrip("0") if toks[j].isdigit() else toks[j]
                if b:
                    keys.add(a2 + "_" + b)
    return " ".join(sorted(keys))


def prepare_texts(df):
    """All per-record text columns Stage 2 needs, computed once."""
    out = pd.DataFrame({"entity_id": df["entity_id"].values})
    out["cname"] = clean_name(df["business_name"]).values
    out["caddr"] = clean_addr(df["business_address"]).values
    out["digits"] = out["caddr"].str.findall(r"\b\d+\b").str.join(" ")
    out["nospace"] = out["cname"].str.replace(" ", "", regex=False)
    out["akeys"] = [addr_key_string(a) for a in out["caddr"]]
    out["nonlatin"] = np.array([any(ord(ch) > 0x024F for ch in s)
                                for s in df["business_name"]], dtype=np.int8)
    return out


# =============================================================================
# PART 1 — Prepare and save text columns, aligned to qids / pids
# =============================================================================
print("=" * 70)
print("PART 1  Text preparation")
print("=" * 70)

t = time.time()
recs = pd.concat([load_src(DATA_ROOT / "test" / f"test_source{i}.tsv") for i in (1, 2, 3)],
                 ignore_index=True)
row_of = pd.Index(recs["entity_id"])
log(f"loaded {len(recs):,} test records in {time.time()-t:.0f}s")

COUNTRIES = sorted(d.name for d in CKPT.iterdir() if (d / "pairs.npz").exists())
prep_rows, prep_secs = 0, 0.0
log(f"countries with blocking output: {COUNTRIES}")

for country in COUNTRIES:
    out_dir = PREP / country
    out_dir.mkdir(parents=True, exist_ok=True)
    for side in ("q", "p"):
        path = out_dir / f"{side}.parquet"
        if path.exists():
            log(f"[{country}] {side}.parquet exists — skipping")
            continue
        ids = np.load(CKPT / country / f"{side}ids.npy", allow_pickle=True)
        rows = row_of.get_indexer(ids)
        assert (rows >= 0).all(), f"{country}/{side}: {int((rows < 0).sum())} ids not in test files"
        t = time.time()
        prepared = prepare_texts(recs.iloc[rows])
        assert (prepared["entity_id"].values == ids).all(), "alignment check failed"
        prep_rows += len(ids)
        prep_secs += time.time() - t
        prepared.to_parquet(path, index=False)
        log(f"[{country}] {side}: {len(ids):,} rows prepared in {time.time()-t:.0f}s "
            f"-> {path.stat().st_size/1e6:.0f} MB")
        del prepared
        gc.collect()

del recs, row_of
gc.collect()
mem("after text preparation")


# =============================================================================
# PART 2 — Real-speed check on this machine, with real test pairs
# =============================================================================
print("\n" + "=" * 70)
print("PART 2  Speed check on real test pairs")
print("=" * 70)

big = max(COUNTRIES, key=lambda c: len(np.load(CKPT / c / "pids.npy", allow_pickle=True)))
R = {}
if prep_rows:
    R["prep_rows_s"] = prep_rows / max(prep_secs, 1e-9)
else:
    R["prep_rows_s"] = 40_000     # prep files already existed; conservative placeholder
t = time.time()
d = np.load(CKPT / big / "pairs.npz")
q_all, p_all = d["q"], d["p"]
S = np.column_stack([d["l1"], d["l2"], d["l3"], d["ak"].astype(np.float16)])
R["npz_rows_s"] = len(q_all) / (time.time() - t)
log(f"benchmark country: {big} ({len(q_all):,} pairs)")

t = time.time()
qt = pd.read_parquet(PREP / big / "q.parquet")
pt = pd.read_parquet(PREP / big / "p.parquet")
R["parquet_rows_s"] = (len(qt) + len(pt)) / (time.time() - t)


def group_features(q, S):
    """Scores relative to each S1's own candidate list (q must be sorted).
    Scale-free features like rank and gap-to-best transfer across countries."""
    starts = np.flatnonzero(np.r_[True, q[1:] != q[:-1]])
    sizes = np.diff(np.r_[starts, len(q)])
    cols = []
    for j in range(S.shape[1]):
        s = S[:, j].astype(np.float32)
        gmax = np.repeat(np.maximum.reduceat(s, starts), sizes)
        order = np.lexsort((-s, q))
        rank = np.empty(len(q), np.float32)
        rank[order] = np.arange(len(q)) - np.repeat(starts, sizes)
        cols += [s, gmax - s, rank]
    cols += [(S[:, :3] > 0).sum(1) + (S[:, 3] > 0), np.repeat(sizes, sizes)]
    return np.column_stack(cols).astype(np.float32)


n = min(BENCH_GROUP_ROWS, len(q_all))
t = time.time()
Xg = group_features(q_all[:n], S[:n])
R["group_rows_s"] = n / (time.time() - t)

# Pre-filter model speed (labels are random: this measures speed only).
rng = np.random.default_rng(0)
m = lgb.train({"objective": "binary", "num_leaves": 31, "learning_rate": 0.1,
               "min_data_in_leaf": 5, "min_gain_to_split": 0, "verbose": -1},
              lgb.Dataset(Xg[:200_000], rng.integers(0, 2, 200_000)), 150)
k = min(5_000_000, n)
t = time.time()
pf = m.predict(Xg[:k])
R["prefilter_rows_s"] = k / (time.time() - t)

t = time.time()
qq = q_all[:k]
order = np.lexsort((-pf, qq))
starts = np.flatnonzero(np.r_[True, qq[order][1:] != qq[order][:-1]])
rank = np.arange(k) - np.repeat(starts, np.diff(np.r_[starts, k]))
_ = order[rank < 20]
R["topn_rows_s"] = k / (time.time() - t)
del Xg, pf
gc.collect()

# String features on real pairs.
n = min(BENCH_PAIRS, len(q_all))
qa, pa = q_all[:n], p_all[:n]
t = time.time()
cols = {c: (qt[c].to_numpy()[qa].tolist(), pt[c].to_numpy()[pa].tolist())
        for c in ("cname", "caddr", "digits", "nospace", "akeys")}
R["gather_pairs_s"] = n / (time.time() - t)
SCORERS = [("cname", "ratio", fuzz.ratio), ("cname", "token_set", fuzz.token_set_ratio),
           ("cname", "token_sort", fuzz.token_sort_ratio), ("cname", "partial", fuzz.partial_ratio),
           ("cname", "jaro_winkler", JaroWinkler.normalized_similarity),
           ("nospace", "ratio", fuzz.ratio), ("caddr", "token_set", fuzz.token_set_ratio),
           ("caddr", "token_sort", fuzz.token_sort_ratio), ("caddr", "ratio", fuzz.ratio),
           ("digits", "token_set", fuzz.token_set_ratio), ("akeys", "token_set", fuzz.token_set_ratio)]
t_all = time.time()
for col, nm, sc in SCORERS:
    t = time.time()
    cpdist(cols[col][0], cols[col][1], scorer=sc, workers=-1)
    print(f"    {col:>8} {nm:<13} {n/(time.time()-t)/1e6:6.2f} M pairs/s")
R["string_pairs_s"] = n / (time.time() - t_all)
del cols
gc.collect()

# Main model: training and prediction speed (random data: speed only).
Xm = rng.standard_normal((1_000_000, 32)).astype(np.float32)
ym = (Xm[:, 0] + rng.standard_normal(1_000_000) > 1.5).astype(np.int8)
t = time.time()
mm = lgb.train({"objective": "binary", "num_leaves": 63, "learning_rate": 0.05,
                "feature_fraction": 0.8, "bagging_fraction": 0.8, "bagging_freq": 1,
                "verbose": -1}, lgb.Dataset(Xm, ym), 300)
R["matcher_train_s_per_Mrows_300"] = time.time() - t
t = time.time()
mm.predict(Xm)
R["matcher_rows_s"] = 1_000_000 / (time.time() - t)
del Xm, ym
gc.collect()

print()
for key, v in R.items():
    print(f"  {key:<32} {v:>14,.0f}")


# =============================================================================
# PART 3 — Runtime projection for Step 1 (train) and Step 2 (test)
# =============================================================================
print("\n" + "=" * 70)
print("PART 3  Projected runtimes (from the rates measured above)")
print("=" * 70)

test_pairs = sum(json.load(open(CKPT / c / "stats.json"))["pairs"] for c in COUNTRIES)
test_q = sum(json.load(open(CKPT / c / "stats.json"))["queries"] for c in COUNTRIES)
avg_c = test_pairs / test_q
train_q = int(TRAIN_S1 * 0.30)
train_pairs = int(train_q * avg_c)
fit_q = int(TRAIN_S1 * 0.10)


def minutes(x):
    return x / 60


for N in N_KEEP_OPTIONS:
    kept_test, kept_train, kept_fit = test_q * N, train_q * N, fit_q * N
    step1 = {
        "load pairs": minutes(train_pairs / R["npz_rows_s"]),
        "prepare train texts": minutes((train_q + TRAIN_POOL) / R["prep_rows_s"]),
        "group features": minutes(train_pairs / R["group_rows_s"]),
        "pre-filter train+predict": minutes(train_pairs / R["prefilter_rows_s"]) + 1,
        "keep top-N": minutes(train_pairs / R["topn_rows_s"]),
        "string features": minutes(kept_train / R["string_pairs_s"]),
        "main model training": minutes(kept_fit / 1e6 * R["matcher_train_s_per_Mrows_300"]
                                       * MATCHER_ROUNDS / 300),
        "predict + threshold search": minutes(kept_train * 2 / 3 / R["matcher_rows_s"]) + 1,
    }
    step2 = {
        "load pairs + texts": minutes(test_pairs / R["npz_rows_s"] + 11_701_862 / R["parquet_rows_s"]),
        "group features": minutes(test_pairs / R["group_rows_s"]),
        "pre-filter predict": minutes(test_pairs / R["prefilter_rows_s"]),
        "keep top-N": minutes(test_pairs / R["topn_rows_s"]),
        "string features": minutes(kept_test / R["string_pairs_s"]),
        "main model predict": minutes(kept_test / R["matcher_rows_s"]),
        "write + official validator": minutes(kept_test / 4e6) + minutes(kept_test / 252e6 * 3.6 * 60 * 1.5),
    }
    print(f"\n  keep top-{N} per entity  ->  {kept_test/1e6:.0f}M test pairs scored, "
          f"{kept_train/1e6:.0f}M train pairs")
    for label, parts in (("Step 1 (train)", step1), ("Step 2 (test)", step2)):
        print(f"    {label:<15} ~{sum(parts.values()):5.1f} min   "
              + "  ".join(f"{k} {v:.1f}" for k, v in parts.items()))

print("\n  Step 1 assumes the train pairs are about 30% of train S1 x the test")
print("  average candidates per entity; the real figure comes from the train run.")
mem("final")
log("STEP 0 COMPLETE")